# B2.2 — Analyse des cibles : les fingerprints du catalogue

**Objectif** (plan B, tâche B2.2) : comprendre ce que le modèle devra apprendre, et à quel point la validation est difficile.

1. **Fréquence des bits** : combien sont presque toujours à 0, presque toujours à 1 ? Quelle proportion de bits à 1 ? (important pour la loss du sprint 3)
2. **Fingerprints identiques** : combien de molécules différentes partagent exactement le même fingerprint ?
3. **Nouveauté de la validation** : pour chaque molécule `dev`, la similarité de Tanimoto maximale avec une molécule du train.
4. **Pouvoir discriminant dans la fenêtre de masse** : combien de candidats ressemblent beaucoup à la bonne molécule ?

Fingerprints de référence : Morgan, rayon 2, 2048 bits, binaire (`src/casmi/fingerprints.py`).

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Racine du dépôt : premier dossier parent qui contient pyproject.toml
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))

from casmi.catalog import load_catalog, mass_windows
from casmi.fingerprints import CATALOG_FP_PATH, FP_RADIUS, FP_SIZE, load_catalog_fps

catalog = load_catalog()
fps = load_catalog_fps()
print(f"Catalogue : {len(catalog):,} molécules | fingerprints {fps.shape}, rayon {FP_RADIUS}, {FP_SIZE} bits")
print("Fichier :", CATALOG_FP_PATH)


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="#e5e4e0", linewidth=0.8)
    ax.set_axisbelow(True)

## 1. Fréquence des bits

In [ ]:
bit_freq = fps.mean(axis=0, dtype=np.float64)
bits_per_mol = fps.sum(axis=1, dtype=np.int64)
density = bits_per_mol.mean() / FP_SIZE

print(f"Bits à 1 par molécule : moyenne {bits_per_mol.mean():.1f}, médiane {np.median(bits_per_mol):.0f}, "
      f"min {bits_per_mol.min()}, max {bits_per_mol.max()}")
print(f"Proportion moyenne de bits à 1 : {density:.2%}")
print(f"→ pos_weight « naturel » pour BCEWithLogitsLoss, (1 − p) / p : environ {(1 - density) / density:.0f}")
print()
for label, mask in [
    ("jamais à 1", bit_freq == 0),
    ("à 1 dans moins de 0,1 % des molécules", bit_freq < 0.001),
    ("à 1 dans moins de 1 % des molécules", bit_freq < 0.01),
    ("à 1 dans plus de 50 % des molécules", bit_freq > 0.5),
    ("à 1 dans plus de 99 % des molécules", bit_freq > 0.99),
]:
    print(f"Bits {label:40s} : {mask.sum():5d} sur {FP_SIZE}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(np.arange(1, FP_SIZE + 1), np.sort(bit_freq)[::-1], color="#2a78d6", linewidth=2)
ax.set_yscale("log")
ax.set_xlabel("rang du bit (du plus fréquent au plus rare)")
ax.set_ylabel("fréquence du bit (échelle log)")
ax.set_title("Quelques bits très fréquents, une longue traîne de bits rares", loc="left")
for level, text in [(0.01, "1 % des molécules"), (0.001, "0,1 % des molécules")]:
    ax.axhline(level, color="#52514e", linestyle="--", linewidth=1)
    ax.text(30, level * 1.12, text, va="bottom", color="#52514e", fontsize=9)
style(ax)
plt.show()

## 2. Fingerprints identiques

Deux molécules qui ont exactement le même fingerprint seront **impossibles à départager** par le classement au Tanimoto. C'est surtout grave quand elles ont aussi la même formule, donc la même masse : elles tombent alors toujours dans la même fenêtre de candidats.

In [ ]:
# Une empreinte compacte de chaque ligne pour repérer les doublons
keys = pd.Series([row.tobytes() for row in np.packbits(fps, axis=1)])
group_size = keys.map(keys.value_counts())
same_fp_and_formula = pd.Series(list(zip(keys, catalog["formula"]))).pipe(lambda s: s.map(s.value_counts()))

print(f"Fingerprints distincts : {keys.nunique():,} pour {len(keys):,} molécules")
print(f"Molécules qui partagent leur fingerprint avec au moins une autre : {(group_size > 1).sum():,} ({(group_size > 1).mean():.1%})")
print(f"  dont avec la même formule (même masse, donc indiscernables) : {(same_fp_and_formula > 1).sum():,} "
      f"({(same_fp_and_formula > 1).mean():.1%})")
print(f"Plus grand groupe de molécules au fingerprint identique : {group_size.max()}")

In [ ]:
largest = catalog.loc[group_size == group_size.max(), ["inchikey14", "smiles", "formula"]]
print("Exemple : le plus grand groupe (5 premières molécules)")
largest.head()

## 3. Nouveauté de la validation

On reprend **le même tirage provisoire que le notebook 02** (10 % des molécules, graine 42, dont 2 000 en `dev`). Pour chaque molécule `dev`, on cherche la molécule du **train** la plus proche au sens de Tanimoto.

Tanimoto = bits communs / bits présents dans l'un ou l'autre. Le nombre de bits communs à deux fingerprints est un produit scalaire : on calcule donc toutes les similarités par blocs, avec une multiplication de matrices qui utilise tous les cœurs.

In [ ]:
all_keys = np.sort(catalog["inchikey14"].to_numpy())
rng = np.random.default_rng(42)
val_keys = rng.choice(all_keys, size=int(0.10 * len(all_keys)), replace=False)
dev_keys = rng.choice(np.sort(val_keys), size=2_000, replace=False)

is_val = catalog["inchikey14"].isin(val_keys).to_numpy()
dev_idx = np.flatnonzero(catalog["inchikey14"].isin(dev_keys).to_numpy())
train_idx = np.flatnonzero(~is_val)
print(f"dev : {len(dev_idx):,} molécules | train : {len(train_idx):,} molécules")


def max_tanimoto(query_idx, reference_idx, block=10_000):
    """Pour chaque requête, Tanimoto maximal avec les références, et l'indice de la plus proche."""
    q = fps[query_idx].astype(np.float32)
    q_bits = q.sum(axis=1)
    best = np.zeros(len(query_idx))
    best_ref = np.full(len(query_idx), -1)
    for start in range(0, len(reference_idx), block):
        ref = reference_idx[start : start + block]
        r = fps[ref].astype(np.float32)
        common = q @ r.T
        sim = common / (q_bits[:, None] + r.sum(axis=1)[None, :] - common)
        j = sim.argmax(axis=1)
        better = sim[np.arange(len(q)), j] > best
        best[better] = sim[np.arange(len(q)), j][better]
        best_ref[better] = ref[j[better]]
    return best, best_ref


t0 = time.perf_counter()
novelty, nearest = max_tanimoto(dev_idx, train_idx)
print(f"{len(dev_idx):,} × {len(train_idx):,} similarités calculées en {time.perf_counter() - t0:.0f} s")
print(f"Tanimoto max avec le train : médiane {np.median(novelty):.2f}")
for t in (1.0, 0.9, 0.7, 0.5):
    print(f"  molécules dev avec une voisine du train à Tanimoto ≥ {t} : {(novelty >= t - 1e-9).mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(novelty, bins=np.linspace(0, 1, 41), color="#2a78d6", edgecolor="white", linewidth=0.5)
ax.axvline(np.median(novelty), color="#52514e", linestyle="--", linewidth=1)
ax.text(np.median(novelty), 0.97, f" médiane {np.median(novelty):.2f}".replace(".", ","), transform=ax.get_xaxis_transform(),
        va="top", color="#52514e", fontsize=9)
ax.set_xlabel("Tanimoto maximal avec une molécule du train")
ax.set_ylabel("molécules dev")
ax.set_title("À quel point les molécules de validation sont-elles nouvelles ?", loc="left")
style(ax)
plt.show()

In [ ]:
print("Exemples : molécules dev et leur plus proche voisine du train")
pd.DataFrame({
    "dev": catalog["smiles"].to_numpy()[dev_idx],
    "voisine du train": catalog["smiles"].to_numpy()[nearest],
    "tanimoto": novelty.round(2),
}).sort_values("tanimoto", ascending=False).iloc[[0, 500, 1000, 1500, 1999]]

## 4. Pouvoir discriminant dans la fenêtre de masse

Pour chaque molécule `dev`, on prend ses candidats à **10 ppm** autour de sa masse exacte, dans le catalogue « monde fermé » (train et validation), et on compte ceux qui lui ressemblent beaucoup. Un candidat au fingerprint **identique** (Tanimoto = 1) sera toujours ex æquo avec la bonne réponse, même pour un modèle parfait.

*À coordonner avec C* : son « oracle » (classement avec le vrai fingerprint) part du même calcul.

In [ ]:
masses = catalog["exact_mass"].to_numpy()
lo, hi = mass_windows(masses[dev_idx], masses, ppm=10)

rows = []
for i, a, b in zip(dev_idx, lo, hi):
    cand = np.setdiff1d(np.arange(a, b), [i])
    if len(cand):
        common = (fps[cand] & fps[i]).sum(axis=1)
        sim = common / (fps[cand] | fps[i]).sum(axis=1)
    else:
        sim = np.array([])
    rows.append({
        "candidats": len(cand),
        "concurrent le plus proche": sim.max() if len(sim) else np.nan,
        "≥ 0,8": (sim >= 0.8).sum(),
        "≥ 0,9": (sim >= 0.9).sum(),
        "= 1": (sim >= 1 - 1e-9).sum(),
    })
window = pd.DataFrame(rows)
print(f"Tanimoto avec le concurrent le plus proche : médiane {window['concurrent le plus proche'].median():.2f}")

print(f"Candidats par molécule dev (hors elle-même) : médiane {window['candidats'].median():.0f}")
for col in ["≥ 0,8", "≥ 0,9", "= 1"]:
    print(f"Molécules dev avec au moins un candidat à Tanimoto {col} : {(window[col] > 0).mean():.1%} "
          f"(médiane parmi elles : {window.loc[window[col] > 0, col].median():.0f})")

In [ ]:
closest = window["concurrent le plus proche"].dropna()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.hist(closest, bins=np.linspace(0, 1, 41), color="#2a78d6", edgecolor="white", linewidth=0.5)
for t in (0.8, 0.9):
    ax.axvline(t, color="#52514e", linestyle="--", linewidth=1)
    ax.text(t, 0.97, f" {t:.1f}".replace(".", ","), transform=ax.get_xaxis_transform(), va="top", color="#52514e", fontsize=9)
ax.set_xlabel("Tanimoto entre la bonne molécule et son concurrent le plus proche (fenêtre de 10 ppm)")
ax.set_ylabel("molécules dev")
ax.set_title("Dans la fenêtre de masse, la bonne réponse a rarement un sosie", loc="left")
style(ax)
plt.show()

## Conclusions

**1. Des cibles très déséquilibrées.** Une molécule a en moyenne 46 bits à 1 sur 2048, soit **2,25 %**. Les 2048 bits servent tous, mais 1 174 d'entre eux (57 %) sont à 1 dans moins de 1 % des molécules, et seuls 13 bits le sont dans plus de la moitié. Un modèle qui prédirait « tout à 0 » aurait donc 97,8 % de bits justes et ne servirait à rien : au sprint 3, il faudra suivre le **Tanimoto** et la précision et le rappel des bits à 1, pas l'exactitude. Le `pos_weight` « naturel » vaudrait environ 43 ; la bonne valeur sera à tester au sprint 4.

**2. Quelques molécules indiscernables.** Il y a 264 446 fingerprints distincts pour 275 810 molécules. 4,9 % des molécules partagent leur fingerprint avec une autre, et **1,6 % ont aussi la même formule**, donc la même masse : aucun classement au Tanimoto ne pourra les départager. Le plus grand groupe (360 molécules) est formé de lipides qui ne diffèrent que par la longueur de leurs chaînes : un fingerprint binaire de rayon 2 ne compte pas les CH₂ répétés. Un **fingerprint à comptage** (piste du sprint 4) pourrait les séparer.

**3. Une validation déjà exigeante.** Le Tanimoto maximal d'une molécule `dev` avec le train vaut **0,65 en médiane**. Seules 6,7 % des molécules `dev` ont un quasi-jumeau dans le train (Tanimoto ≥ 0,9), dont 4,5 % au fingerprint identique. 35,4 % ont une voisine à au moins 0,7, et 92,8 % à au moins 0,5. Le split par molécule n'est donc pas « trop facile ». **Question à trancher avec A** : faut-il en plus un split par scaffold, plus exigeant ? Il pourrait servir de contrôle de robustesse, sans remplacer le split actuel.

**4. Dans la fenêtre de masse, la bonne réponse a rarement un sosie.** Sur une médiane de 60 candidats à 10 ppm, le concurrent le plus proche de la bonne molécule n'est qu'à **0,34 de Tanimoto en médiane**. Seules 3,9 % des molécules ont un candidat à au moins 0,8, et 1,5 % un candidat au fingerprint identique (ex æquo forcé). Avec le vrai fingerprint, la bonne réponse serait donc presque toujours classée première : l'« oracle » de C devrait être proche du maximum. **Toute la difficulté sera de bien prédire le fingerprint à partir du spectre.**